# Tensorflow
- Efficiently execute and graph of computations by breaking up the graph into chunks to be run in parallel across CPUs and GPUs. 
- Integrates nicely with Scikit-learn (TF.Learn)
- Provides highly optimized code to search for hyperparameters aiming to minimize a cost function (automatic differentiating) 

In [3]:
import tensorflow as tf
import sklearn
import numpy as np
import pandas
import torch
import sklearn.datasets
import sklearn.preprocessing
import helpers
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, precision_recall_curve, roc_auc_score, f1_score, make_scorer, auc, average_precision_score
import datetime

**Allocating a fixed memory limit to the GPU**

In [4]:
print(tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(
                memory_limit=4096)]  # Set memory limit to 4GB
        )
        print("GPU memory limit set to 4096MB")
    except RuntimeError as e:
        print(e)

2.10.1
GPU memory limit set to 4096MB


**Create a session**
- Session is a runtime environment that executes computational graphs
- Allow TensorFlow to manage operations and to allocate the needed resources efficiently

Note that TensorFlow2 does not need any session as its 'eager execution' by default

In [5]:
#sess = tf.compat.v1.Session()
#sess
X = tf.Variable(3, name='X')
Y = tf.Variable(4, name='Y')
f = X*X*Y+Y+2 #Eager execution right away

**Displaying computational graph result**

In [6]:
f.numpy()

42

**Visualizing the computational graph**

In [10]:
logdir = "logs/graph/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(logdir)
X = tf.Variable(3, name='X')
Y = tf.Variable(4, name='Y')

@tf.function
def function(x, y):
    return x*x*y+y+2

tf.summary.trace_on(graph=True, profiler=True)  # Enable tracing
res = function(X, Y)
with writer.as_default():
    tf.summary.trace_export(name="my_function_graph", step=0, profiler_outdir=logdir) # Export the trace
print(f"Graph logged in {logdir}")

Graph logged in logs/graph/20250318-211510


**Terminology**
- Operations (Ops): Takes any number of inputs and produces any number of outputs. 
  - Inputs and outputs are multi-dimensional arrays: TENSORS
    - In Python API, tensors are represented by NumPy ndarrays
- Constants / variables take no inputs: They are considered source ops
- Placeholder nodes: tf.placeholder(datatype,shape()): Deprecated in TensorFlow 2.*
  - If we specify the shape to be None, for any of the components, it means 'any size' (dynamic size)
  - Typically used to pass training data to TensorFlow during batch training
  - Does not perform any computation, and just outputs the data you tell them to output at runtime
    - A must be 2D
  - In TensorFlow 2, just convert the dynamic data into tensors

Autodiff: (tf.gradients(operation, [variables]))
- Mathematically efficient method to compute the gradient
- Takes an operation and a list of variables
  - Computes the gradient of the 'operation' with respect to each of the variable


There is an even more efficient optimizer of Gradient Descent:
- tf.train.GradientDescentOptimizer(learning_rate)
- tf.train.MomentumOptimizer(learning_rate, momentum_rate)

Can actually save model into a file tf.train.Saver()
- Create after all variable nodes finished creating 
- Call save() method during whenever a save is needed
- Call restore() during retrieval

**Name scope: Avoid clustering of thousands of nodes**
- tf.name_scope("scope_name"): produces a 'environment' where related nodes are hidden under. The graphical displays only the scope_name as a whole, and not individual subcomponents

**Shared Variables**
- TensorFlow has the option to create shared variable (get_variable()) if the variable doesnt exist yet, otherwise it reuses it.
  - Shared variables are defined under the current defined variable_scope() 
    - tf.variable_scope("scope_name", reuse=True)

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing

In [24]:
housing= fetch_california_housing()
m, n  = housing.data.shape
housing_data_plus_bias = np.c_[np.ones((m, 1)), housing.data]
X = tf.constant(housing_data_plus_bias, dtype=tf.float32, name="X")
Y = tf.constant(housing.target.reshape(-1,1), dtype=tf.float32, name="Y")
XT = tf.transpose(X)
# theta = (XT*X)^-1 * XT
theta = tf.matmul(tf.matmul(tf.linalg.inv(tf.matmul(XT, X)), XT), Y)

theta.numpy()

array([[-4.1628044e+01],
       [ 4.6098977e-01],
       [ 1.1416475e-02],
       [-1.2757403e-01],
       [ 7.0443976e-01],
       [-7.1159102e-06],
       [-4.0764478e-03],
       [-3.8447931e-01],
       [-4.2850903e-01]], dtype=float32)

**Manual Gradient Descent Computation**

In [65]:
epochs = 1000
learning_rate = 0.01
scaler = StandardScaler()
scaled_housing_data_plus_bias = scaler.fit_transform(housing_data_plus_bias)
X = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32)
y = tf.constant(housing.target.reshape(-1, 1), dtype=tf.float32, name="y")
theta = tf.Variable(tf.random.uniform([n + 1, 1], -1.0, 1.0), name="theta")

In [59]:
for epoch in range(epochs):
    with tf.GradientTape() as tape:
        y_pred = tf.matmul(X, theta, name="predictions")
        error = y_pred - y
        mse = tf.reduce_mean(tf.square(error), name="mse")
        
    # Manual gradient computation************************
    gradients = 2/m * tf.matmul(tf.transpose(X), error)
    #****************************************************
    
    theta.assign(theta - learning_rate * gradients)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, MSE = {mse.numpy()}")
best_theta = theta.numpy()

Epoch 0, MSE = 7.487589359283447
Epoch 100, MSE = 4.883509159088135
Epoch 200, MSE = 4.839156150817871
Epoch 300, MSE = 4.82972526550293
Epoch 400, MSE = 4.823208332061768
Epoch 500, MSE = 4.8183770179748535
Epoch 600, MSE = 4.8147711753845215
Epoch 700, MSE = 4.812068462371826
Epoch 800, MSE = 4.810034275054932
Epoch 900, MSE = 4.808496952056885


**Using autodiff method**

In [61]:
for epoch in range(epochs):
    with tf.GradientTape() as tape:
        y_pred = tf.matmul(X, theta, name="predictions")
        error = y_pred - y
        mse = tf.reduce_mean(tf.square(error), name="mse")
        
    # autodiff method *****************************   
    gradients = tape.gradient(mse, theta)
    # *********************************************
    
    theta.assign(theta - learning_rate * gradients)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, MSE = {mse.numpy()}")
best_theta = theta.numpy()

Epoch 0, MSE = 10.880767822265625
Epoch 100, MSE = 4.9803643226623535
Epoch 200, MSE = 4.886017799377441
Epoch 300, MSE = 4.86198616027832
Epoch 400, MSE = 4.845653057098389
Epoch 500, MSE = 4.833898067474365
Epoch 600, MSE = 4.825410842895508
Epoch 700, MSE = 4.81928014755249
Epoch 800, MSE = 4.814850330352783
Epoch 900, MSE = 4.811649322509766


**Using a gradient descent optimizer**
- To assign new theta value

In [ ]:
optimizer = tf.optimizers.SGD(learning_rate=learning_rate)

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        y_pred = tf.matmul(X, theta, name="predictions")
        error = y_pred - y
        mse = tf.reduce_mean(tf.square(error), name="mse")
        
    # autodiff method *****************************
    gradients = tape.gradient(mse, theta)
    # *********************************************
    
    # SGD optimizer to apply the gradient ************
    optimizer.apply_gradients([(gradients, theta)])
    # ************************************************
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, MSE = {mse.numpy()}")
best_theta = theta.numpy()

Epoch 0, MSE = 6.09678316116333
Epoch 100, MSE = 5.0086588859558105
Epoch 200, MSE = 4.950314998626709
Epoch 300, MSE = 4.913908004760742
Epoch 400, MSE = 4.887077808380127
Epoch 500, MSE = 4.867067813873291
Epoch 600, MSE = 4.852077484130859
Epoch 700, MSE = 4.840798377990723
Epoch 800, MSE = 4.832274436950684
Epoch 900, MSE = 4.825803279876709


In [ ]:
# Convert data to tf.data.Dataset for batching
dataset = tf.data.Dataset.from_tensor_slices((X, y))
batch_size = 64
dataset = dataset.shuffle(buffer_size=100).batch(batch_size)

# Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(X.shape[1],))
])

optimizer = tf.optimizers.SGD(learning_rate=0.01)

for epoch in range(100):
    epoch_loss = 0.0
    num_batches = 0
    for X_batch, y_batch in dataset:
        with tf.GradientTape() as tape:
            y_pred = model(X_batch)
            # Mean Squared Error
            loss = tf.reduce_mean(tf.square(y_pred - y_batch))
            epoch_loss += loss
            num_batches += 1
        
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss = {epoch_loss / num_batches}")

Epoch 0, Loss = 1.0278865098953247
Epoch 10, Loss = 0.5153418183326721
Epoch 20, Loss = 0.5252722501754761
Epoch 30, Loss = 0.5203554630279541
Epoch 40, Loss = 0.5241612195968628
Epoch 50, Loss = 0.5348238348960876
Epoch 60, Loss = 0.5205517411231995
Epoch 70, Loss = 0.5316696763038635
Epoch 80, Loss = 0.5112767219543457
Epoch 90, Loss = 0.5155829787254333
